# 03  Progression per compartment + topographic specificity

**Notebook id**: `03_progression_sites` → vault `03.4-progression-par-site.md`

**Full disclosure** of all six compartments (point C.3): the signal is confined to the **patellofemoral** block. We (1) test each compartment (cyclops vs meniscus) with Mann–Whitney + Cliff δ and BH-FDR (q = 0.10) and bar-plot the worsening %, then (2) show the **within-patient** topographic specificity  Δ`lesion_pf` vs Δ`lesion_ft` paired Wilcoxon.

In [ ]:
import sys
from pathlib import Path

current = Path().absolute().parent
sys.path.insert(0, (current / "src").as_posix())


In [ ]:
# --- Setup (idempotent, fresh-kernel reproducible) ---
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from constants import (
    RANDOM_SEED,
    SITES,
    SITES_PF,
    SITES_FT,
    SITES_BINARY,
    SITES_ORDINAL,
    GROUPS,
    BLOCKS,
    N_TOTAL,
    N_TOTAL_ANALYSABLE,
    N_MENISCUS,
    N_CYCLOPS,
    SCORE_MAX,
    SCORE_MAX_COLLAPSED,
)
import loaders
import preprocessing as pp
import tests_freq as tf
import reporting as rpt
import bayes_models as bm
import viz

np.random.seed(RANDOM_SEED)
viz.set_pub_style()


In [ ]:
# --- Load & preprocess (canonical pipeline) ---
df = loaders.load_combined()
df = pp.apply_date_hygiene(df)  # composite-key (group, anonyme) date hygiene
df = pp.add_derived(df)  # lesion_pf/ft, female, deltas, worsened_pf, ...
wide = pp.to_wide(df)  # one row per patient (group, anonyme)
patient = pp.to_patient(df)  # static covariates per patient

# Patient-level covariates joined onto the wide outcomes (for H3 / sensitivity).
_cov = [
    c
    for c in [
        "group",
        "anonyme",
        "female",
        "sexe",
        "pivot_pivot_contact",
        "travail_physique",
        "tabac",
        "age_at_trauma",
        "imc",
        "taille",
        "poids",
    ]
    if c in patient.columns
]
merged = wide.merge(patient[_cov], on=["group", "anonyme"], how="left")

print(
    "long:",
    df.shape,
    "| wide:",
    wide.shape,
    "| patient:",
    patient.shape,
    "| merged:",
    merged.shape,
)
# Composite-key sentinel: 19 Anonyme ids are reused across the two sheets.
assert (df.groupby(["group", "anonyme"]).size() == 2).all(), "composite key broken"


## 1. Per-compartment progression (cyclops vs meniscus) + BH-FDR

Direction of progression on each compartment Δ; PTI/CFI are binary (grade ≥ 2 never observed). BH-FDR controls the family of 6 tests.

In [ ]:
rows = []
pvals = []
for s in SITES:
    col = f"delta_{s}"
    if col not in wide.columns:
        continue
    c = wide.loc[wide.group == "cyclops", col].dropna().astype(float).values
    m = wide.loc[wide.group == "meniscus", col].dropna().astype(float).values
    r = tf.mwu_with_effects(c, m, n_boot=2000, seed=RANDOM_SEED)
    rows.append(
        dict(
            compartment=s,
            block="PF" if s in SITES_PF else "FT",
            measurement="binary" if s in SITES_BINARY else "ordinal",
            worsened_pct_cyc=round(float((c > 0).mean() * 100), 1),
            worsened_pct_men=round(float((m > 0).mean() * 100), 1),
            cliff_delta=round(float(r["cliffs_delta"]), 4),
            mwu_p=round(float(r["pvalue"]), 4),
        )
    )
    pvals.append(r["pvalue"])
per_comp = pd.DataFrame(rows)
bh = tf.bh_fdr(pvals, q=0.10)
per_comp["bh_p_adj"] = [round(float(x), 4) for x in bh["pvals_corrected"]]
per_comp["bh_reject"] = bh["reject"]
print(per_comp.to_string(index=False))


## 2. Per-compartment worsening bars (PF block vs FT block)

In [ ]:
fig = viz.per_compartment_bars(per_comp)
fig


## 3. Topographic specificity  within-patient Δ`lesion_pf` vs Δ`lesion_ft`

Paired Wilcoxon within the cyclops group: does the PF block worsen more than the FT block in the **same** patient? (Mechanistic specificity, point B.)

> **Read as exploratory.** The FT block is **not** equivalent at baseline (notebook `01`, §2c: TOST p = 0.186, SMD +0.26), so this PF−FT contrast is hypothesis-generating, not a clean causal split: a baseline FT excess in the cyclops group plus regression-to-mean could blunt an FT signal and make the effect look more PF-specific than it truly is.

In [ ]:
spec = tf.paired_pf_vs_ft(wide)
for k, v in spec.items():
    print(f"  {k:22s} = {v}")
print()
print(rpt.format_test_result(spec, "wilcoxon"))


In [ ]:
fig = viz.topographic_specificity(
    wide,
    n_pf_worsened=spec["n_pf_worsened"],
    n_ft_worsened=spec["n_ft_worsened"],
    rank_biserial=spec["rank_biserial"],
    wilcoxon_p=spec["pvalue"],
)
fig


## Sanity asserts

In [ ]:
assert len(per_comp) == 6  # six compartments fully disclosed
# PF compartments should worsen more in cyclops than the FT compartments do.
pf_mean = per_comp.loc[per_comp.block == "PF", "worsened_pct_cyc"].mean()
ft_mean = per_comp.loc[per_comp.block == "FT", "worsened_pct_cyc"].mean()
assert pf_mean > ft_mean, (pf_mean, ft_mean)
print("Per-compartment + specificity asserts passed.")
